# Lab 12: CSP Backtracking for Smart Film Production Casting

## Overview

In this lab, you will implement a Constraint Satisfaction Problem (CSP) solver using:

- Basic Backtracking
- Backtracking with Inference
- Minimum Remaining Values (MRV)
- Degree Heuristic
- Least Constraining Value (LCV)

The system will assign actors to movie scenes while satisfying production constraints.

You will also:
- Read input from files
- Use Object-Oriented Programming (OOP)
- Produce formatted console and file outputs
- Implement AI search heuristics

---

# Real World Scenario

A streaming company is shooting a futuristic science fiction series.

Each scene requires a specific skill such as:
- action
- comedy
- stunt
- sci-fi

Actors possess different skills.

The AI system must assign actors to scenes while ensuring:
- each scene gets a compatible actor
- one actor is not assigned to multiple conflicting scenes
- constraints are satisfied efficiently

This simulates:
- AI scheduling systems
- production planning software
- resource allocation problems

---

# Input File Format

You will use two input files.

## actors.txt

Each line contains:

ActorName: skill1,skill2,skill3

Example:

Ava: action,english

---

## scenes.txt

Each line contains:

SceneName: required_skill

Example:

Scene1: action

---


# Expected Learning Outcomes

By the end of this lab students will be able to:

- Implement recursive backtracking
- Apply CSP heuristics
- Use Forward Checking inference
- Design OOP-based AI systems
- Read and write files in Python
- Solve real-world scheduling problems

## Step 01: Make Imports

In [23]:
from collections import defaultdict

## Step 02: Run Input Handler (Do NOT Modify)

In [24]:
class InputHandler:

    @staticmethod
    def load_data(actor_file, scene_file):

        actors = {}
        scenes = {}

        with open(actor_file, "r") as f:

            for line in f.readlines():

                parts = line.strip().split(": ")

                actor = parts[0]

                skills = parts[1].split(",")

                actors[actor] = skills

        with open(scene_file, "r") as f:

            for line in f.readlines():

                parts = line.strip().split(":")

                scene = parts[0]

                requirement = parts[1].strip()

                scenes[scene] = requirement

        return actors, scenes

## Step 03: Create CSP Class
In this step you will:

- store variables
- store domains
- store constraints
- implement consistency checking

In [25]:
class FilmCSP:

    def __init__(self):

        self.variables = []

        self.domains = {}

        self.constraints = defaultdict(list)

    def add_variable(self, variable, domain):

        self.variables.append(variable)
        self.domains[variable] = domain

    def add_constraint(self, var1, var2):

        self.constraints[var1].append(var2)
        self.constraints[var2].append(var1)


    def is_consistent(self, variable, value, assignment):

        for neighbor in self.constraints[variable]:
            if neighbor in assignment and assignment[neighbor] == value:
                return False

## Step 04: Implement MRV Heuristic

In this step you will:

- select the variable with the smallest remaining domain
- reduce unnecessary branching

In [26]:
class MRVHeuristic:

    @staticmethod
    def select_unassigned_variable(csp, assignment):

        unassigned = [
            var for var in csp.variables
            if var not in assignment
        ]

        return min(unassigned, key=lambda var: len(csp.domains[var]))

## Step 05: Implement Degree Heuristic

In this step you will:

- choose the variable connected to most neighbors
- prioritize difficult variables first

In [27]:
class DegreeHeuristic:

    @staticmethod
    def select_variable(csp, assignment):

        unassigned = [
            var for var in csp.variables
            if var not in assignment
        ]

        return max(
            unassigned,
            key=lambda var: len([
                neighbor for neighbor in csp.constraints[var]
                if neighbor not in assignment
            ])
        )

## Step 06: Implement Least Constraining Value (LCV)

In this step you will:

- order values by least restriction
- try least constraining values first

In [28]:
class LCVHeuristic:

    @staticmethod
    def order_values(variable, csp, assignment):

        def count_conflicts(value):

            conflicts = 0

            for neighbor in csp.constraints[variable]:

                if neighbor not in assignment:

                    for neighbor_value in csp.domains[neighbor]:

                        if neighbor_value == value:
                            conflicts += 1

            return conflicts

        return sorted(csp.domains[variable], key=count_conflicts)

## Step 07: Implement Forward Checking Inference

In this step you will:

- remove inconsistent neighboring values
- apply inference after assignments

In [29]:
class Inference:

    @staticmethod
    def forward_checking(csp, variable, value, assignment):

        for neighbor in csp.constraints[variable]:

            if neighbor not in assignment:

                remaining = [
                    v for v in csp.domains[neighbor]
                    if v != value
                ]

                if len(remaining) == 0:
                    return False

        return True

## Step 08: Implement Backtracking Solver

In this step you will implement:

- recursive backtracking
- assignment generation
- inference integration
- backtracking removal

In [30]:
class BacktrackingSolver:

    def backtrack(self, assignment, csp):

        if len(assignment) == len(csp.variables):
            return assignment

        variable = MRVHeuristic.select_unassigned_variable(
            csp,
            assignment
        )

        for value in LCVHeuristic.order_values(
            variable,
            csp,
            assignment
        ):

            if csp.is_consistent(variable, value, assignment):

                assignment[variable] = value

                if Inference.forward_checking(
                    csp,
                    variable,
                    value,
                    assignment
                ):

                    result = self.backtrack(assignment, csp)

                    if result:
                        return result

                del assignment[variable]

        return None

## Step 09: Output Writer (Do NOT Modify)

In [31]:
class OutputWriter:

    @staticmethod
    def write_output(solution):

        with open("output.txt", "w") as f:

            if solution is None:

                text = "No valid casting found."

                print(text)

                f.write(text)

            else:

                print("\nFinal Scene Casting:\n")

                f.write("Final Scene Casting:\n\n")

                for scene, actor in solution.items():

                    line = f"{scene} -> {actor}"

                    print(line)

                    f.write(line + "\n")

        print("\nResults written to output.txt")

## Step 11: Main Method (Do NOT Modify)

In [32]:
def main():

    actors, scenes = InputHandler.load_data(
        "actors.txt",
        "scenes.txt"
    )

    csp = FilmCSP()

    for scene in scenes:

        valid_actors = []

        for actor in actors:

            if scenes[scene] in actors[actor]:

                valid_actors.append(actor)

        csp.add_variable(scene, valid_actors)

    solver = BacktrackingSolver()

    solution = solver.backtrack({}, csp)

    OutputWriter.write_output(solution)


if __name__ == "__main__":
    main()

No valid casting found.

Results written to output.txt


## Step 12: Questions

1. Why is CSP useful in film production systems?

2. How does MRV reduce search complexity?

3. Why is Least Constraining Value effective?

4. What is the role of Forward Checking?

5. How can this system scale for large productions?

In [ ]:
#Answers
# 1.
# CSP is useful in film production because it helps assign actors,
# scenes, and schedules while satisfying constraints such as
# actor availability, budget, and conflict avoidance.

# 2.
# MRV reduces search complexity by selecting the variable with
# the fewest remaining legal values first, reducing branching
# and detecting failures early.

# 3.
# Degree heuristic selects the variable involved in the largest
# number of constraints, helping reduce future conflicts.

# 4.
# LCV chooses the value that eliminates the fewest choices for
# neighboring variables, increasing chances of success.

# 5.
# Forward checking improves efficiency by removing invalid
# future values immediately after an assignment.